# Plot 4: CARROT vs Routerbench vs Zero Router on SPROUT

Loads pre-computed predictions from `../data/sprout/IBMMIX_o3mini.npy` (generated by the RoBERTa models `CARROT-LLM-Routing/Perfo3` and `CARROT-LLM-Routing/costo3` on the SPROUT-o3mini test set). No training required.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('../plots', exist_ok=True)

data = np.load('../data/sprout/IBMMIX_o3mini.npy', allow_pickle=True).item()
all_models = [k for k in data.keys() if k not in ('prompts', 'categories')]
print(f'{len(all_models)} models:', all_models)

In [ ]:
# Stack per-model arrays into (n_prompts, n_models) matrices
scores = np.stack([np.asarray(data[m]['actual perf']) for m in all_models], axis=1)
costs = np.stack([np.asarray(data[m]['actual cost']) for m in all_models], axis=1)
p_logits = np.stack([np.asarray(data[m]['predicted perf']) for m in all_models], axis=1)
p_costs = np.stack([np.asarray(data[m]['predicted cost']) for m in all_models], axis=1)
p_scores = 1 / (1 + np.exp(-p_logits))  # sigmoid of logits

# Static per-model average cost (Routerbench uses this instead of predicted cost)
avg_costs = costs.mean(axis=0)

print('scores', scores.shape, 'costs', costs.shape, 'p_scores', p_scores.shape, 'p_costs', p_costs.shape)

In [ ]:
# Route under two rules:
#   CARROT      : argmax_m (1-lam)*p_scores - lam*p_costs*100     (per-prompt predicted cost)
#   Routerbench : argmax_m (1-lam)*p_scores - lam*avg_costs*100   (static per-model mean cost)
lamb_range = np.arange(0, 1.001, 0.01)

router_cost = np.zeros((scores.shape[0], lamb_range.shape[0]))
router_perf = np.zeros_like(router_cost)
base_cost = np.zeros_like(router_cost)
base_perf = np.zeros_like(router_cost)

for i, lam in enumerate(lamb_range):
    carrot_idx = ((1 - lam) * p_scores - lam * p_costs * 100).argmax(axis=1, keepdims=True)
    rb_idx = ((1 - lam) * p_scores - lam * avg_costs[None, :] * 100).argmax(axis=1, keepdims=True)

    router_perf[:, i] = np.take_along_axis(scores, carrot_idx, axis=1).reshape(-1)
    router_cost[:, i] = np.take_along_axis(costs, carrot_idx, axis=1).reshape(-1)
    base_perf[:, i] = np.take_along_axis(scores, rb_idx, axis=1).reshape(-1)
    base_cost[:, i] = np.take_along_axis(costs, rb_idx, axis=1).reshape(-1)

In [ ]:
from matplotlib.ticker import MaxNLocator

fig, ax = plt.subplots(1, 1, figsize=(4.3, 4.3))
markers = ['o', 's', 'D', '^', 'v', 'p', '*', 'x', '+', 'h', 'H', 'd', '>', 'P']

cost_mean = costs.mean(axis=0)
scores_mean = scores.mean(axis=0)

# Zero router: line through 3 small-model points, sorted by cost ascending so the line is monotone
zero_names = ['wxai-granite-3-8b-instruct-8k-max-tokens', 'openai-gpt-4o-mini', 'openai-o3-mini']
zero_idx = [all_models.index(m) for m in zero_names if m in all_models]
zero_idx = sorted(zero_idx, key=lambda i: cost_mean[i])
zerorouters = zero_idx

ax.errorbar(router_cost.mean(axis=0), router_perf.mean(axis=0),
            linestyle='--', linewidth=1, alpha=1, c='orange', label='CARROT')
ax.errorbar(base_cost.mean(axis=0), base_perf.mean(axis=0),
            linestyle='--', linewidth=1, alpha=1, c='blue', label='Routerbench')
ax.errorbar(cost_mean[zerorouters], scores_mean[zerorouters],
            linestyle='--', linewidth=1, alpha=1, c='grey', label='zero router')

# Skip labels for the llama-3-3 and llama-3-1 family (they crowd the midrange)
SKIP_LABEL_MODELS = {'wxai-llama-3-3-70b-instruct', 'wxai-llama-3-1-70b-instruct',
                     'wxai-llama-3-1-8b-instruct'}

for i, m in enumerate(all_models):
    x, y = cost_mean[i], scores_mean[i]
    ax.scatter([x], [y], marker=markers[i % len(markers)])
    if m not in SKIP_LABEL_MODELS:
        ax.annotate(m, (x, y), size=5.5)

ax.set_xlabel('Cost Per Query, $')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.60, 0.95)
ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
ax.legend(loc='lower right')
ax.grid(True)
fig.savefig('../plots/sprout_vs_rb_zero.pdf', bbox_inches='tight', dpi=400, transparent=False)
plt.show()

In [ ]:
def AUC(cost, perf):
    return -np.trapz(perf, cost) / (np.max(cost) - np.min(cost))

print('CARROT AUC:      ', AUC(router_cost.mean(axis=0), router_perf.mean(axis=0)))
print('Routerbench AUC: ', AUC(base_cost.mean(axis=0), base_perf.mean(axis=0)))
print('Zero Router AUC: ', AUC(cost_mean[zerorouters], scores_mean[zerorouters]))